In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

In [ ]:
print("Verificando dataset...")
path = kagglehub.dataset_download("msambare/fer2013")
caminho_treino = os.path.join(path, "train")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Treinando usando: {device}")



Verificando dataset...
Using Colab cache for faster access to the 'fer2013' dataset.
Treinando usando: cuda


In [ ]:
print("Preparando as imagens...")

#pipeline de transformacao para RGB, 224 x 224px, transforma na matriz tensor e ajustando as imagens pro padrao EfficientNet
data_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

#carrega as imagens do folder e divide elas em 2 Threads para ir buscando em samples de 64 e ir servindo a gpu, e aplica a trasnformacao em cada batch
train_dataset = datasets.ImageFolder(root=caminho_treino, transform=data_transforms)
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True,num_workers=2)
print(f"Total de imagens carregadas: {len(train_dataset)}")

print("Baixando o modelo EfficientNetB0...")
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

#desativa o aprendizado do gradiente para nao alterar aprendizados base e nao demorar tanto no aprendizado
for parametro in model.parameters():
    parametro.requires_grad = False

#troca a camada base para apenas as 7 categorias e deixa ela liberada pra o aprendizado para aprender
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 7)# 7 emoções

for name, param in model.named_parameters():
    if "features.7" in name or "features.8" in name:
        param.requires_grad = True # Libera para aprender
    if "classifier" in name:
        param.requires_grad = True

model = model.to(device)

#definicao do erro, ele é melhor para tarefas de Multipla Escolha e é o melhor Otimizador de erro e passa apenas os parametros da camada de aprendizado para nao baguncar o resto
criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001) #evita reserva de memoria inutil para parametros congelados
y_true = []
y_pred = []
nomes_classes = train_dataset.classes

epochs = 5
print("🚀 Iniciando o treinamento! Vai levar alguns minutos...\n")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    acertos = 0
    total = 0

    for i, (imagens,gabaritos) in enumerate(train_loader):
        #transforma da memória Ram para memória VRAM
        imagens, gabaritos = imagens.to(device), gabaritos.to(device) 

        #calcula o erro conforme o chute(foward pass) e depois faz o backpropagation e aplica os erros otimizando os pesos 
        otimizador.zero_grad()
        outputs = model(imagens)
        loss = criterio(outputs,gabaritos)
        loss.backward()
        otimizador.step()
        running_loss +=loss.item()

        #vai pegar o chute mais parecido e verificar quantos acertaram 
        _, chutado = torch.max(outputs.data,1)
        y_pred.extend(chutado.cpu().numpy())
        y_true.extend(gabaritos.cpu().numpy())
        total += gabaritos.size(0)
        acertos += (chutado == gabaritos).sum().item()

        if i % 100 == 0 and i > 0:
            print(f"   [Época {epoch+1}] Lote {i} processado...")

    loss_final = running_loss/len(train_loader)
    acuracia = 100 * acertos/total
    print(f"✅ Época {epoch+1}/{epochs} | Loss: {loss_final:.4f} | Acurácia: {acuracia:.2f}%\n")
    
torch.save(model.state_dict(), 'efficientnet_emocoes.pth')
print("🎉 Treinamento concluído! Modelo salvo como 'efficientnet_emocoes.pth'")

Preparando as imagens...
Total de imagens carregadas: 28709
Baixando o modelo EfficientNetB0...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 198MB/s]


🚀 Iniciando o treinamento! Vai levar alguns minutos...

   [Época 1] Lote 100 processado...
   [Época 1] Lote 200 processado...
   [Época 1] Lote 300 processado...
   [Época 1] Lote 400 processado...
✅ Época 1/5 | Loss: 1.5437 | Acurácia: 40.19%

   [Época 2] Lote 100 processado...
   [Época 2] Lote 200 processado...
   [Época 2] Lote 300 processado...
   [Época 2] Lote 400 processado...
✅ Época 2/5 | Loss: 1.3054 | Acurácia: 50.38%

   [Época 3] Lote 100 processado...
   [Época 3] Lote 200 processado...
   [Época 3] Lote 300 processado...
   [Época 3] Lote 400 processado...
✅ Época 3/5 | Loss: 1.2007 | Acurácia: 54.53%

   [Época 4] Lote 100 processado...
   [Época 4] Lote 200 processado...
   [Época 4] Lote 300 processado...
   [Época 4] Lote 400 processado...
✅ Época 4/5 | Loss: 1.1335 | Acurácia: 57.81%

   [Época 5] Lote 100 processado...
   [Época 5] Lote 200 processado...
   [Época 5] Lote 300 processado...
   [Época 5] Lote 400 processado...
✅ Época 5/5 | Loss: 1.0728 | Acuráci

In [ ]:
cm = confusion_matrix(y_true, y_pred)

# 5. Plotar a matriz usando Seaborn
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=nomes_classes, 
            yticklabels=nomes_classes)

# Configurações do gráfico
plt.title('Matriz de Confusão - Detecção de Emoções', fontsize=16)
plt.ylabel('Gabarito (Real)', fontsize=12)
plt.xlabel('Previsão do Modelo', fontsize=12)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()

# Mostrar o gráfico
plt.show()